# 05 · Train, compare, and register models with MLflow

In [1]:
import os
import json
import hashlib
import pandas as pd
import mlflow
import mlflow.sklearn
import psycopg2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
from mlflow.tracking import MlflowClient

conn = psycopg2.connect(
    host=os.getenv("DATASET_POSTGRES_HOST", "dataset-postgres"),
    port=os.getenv("DATASET_POSTGRES_PORT", "5432"),
    dbname=os.getenv("DATASET_POSTGRES_DB", "heart_disease"),
    user=os.getenv("DATASET_POSTGRES_USER", "dataset_user"),
    password=os.getenv("DATASET_POSTGRES_PASSWORD", "dataset_pass"),
)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM heart_disease_processed")
        columns = [desc.name for desc in cur.description]
        rows = cur.fetchall()
finally:
    conn.close()
df = pd.DataFrame(rows, columns=columns).sort_values(columns).reset_index(drop=True)
if "target" not in df or df.empty:
    raise ValueError("The processed dataset must contain rows and a target column")

# Stable fingerprint over the rows and column order actually used for training.
dataset_sha256 = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=False).values.tobytes()
    + "|".join(df.columns).encode("utf-8")
).hexdigest()
# Float features keep the logged input schema compatible with missing numeric values.
X = df.drop(columns=["target"]).astype("float64")
y = df["target"].astype("int64")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
EXPERIMENT_NAME = "uci-heart-disease"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow_dataset = mlflow.data.from_pandas(
    df, name="uci-heart-disease-processed", targets="target"
)
print(f"Rows: {len(df)} | features: {X.shape[1]} | sha256: {dataset_sha256}")
print(f"Train rows: {len(X_train)} | test rows: {len(X_test)} | tracking: {TRACKING_URI}")


2026/10/05 04:20:17 INFO mlflow.tracking.fluent: Experiment with name 'uci-heart-disease' does not exist. Creating a new experiment.
2026/10/05 04:20:18 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    expo

Rows: 297 | features: 13 | sha256: f1bf24e70747d13a7196dec484f4b1d7ed3e02fb0a514e1374f88f5cafe7a77a
Train rows: 237 | test rows: 60 | tracking: http://mlflow:5000


## Run 20 experiments

The experiment tests 5 values of `n_estimators` × 4 values of `max_depth`. Fixed parameters and the split are also recorded so each result can be interpreted and reproduced. Metrics use the test set, and each model is saved to the MLflow artifact store (MinIO).

In [2]:
runs = []
for n_estimators in [50, 100, 150, 200, 250]:
    for max_depth in [None, 5, 10, 15]:
        params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": 2,
            "max_features": "sqrt",
            "random_state": 42,
        }
        model = RandomForestClassifier(**params)
        run_name = f"rf-trees-{n_estimators}-depth-{max_depth or 'none'}"
        with mlflow.start_run(run_name=run_name) as run:
            mlflow.log_params(params)
            mlflow.log_params({
                "dataset_source": (
                    f"{os.getenv('DATASET_POSTGRES_HOST', 'dataset-postgres')}:"
                    f"{os.getenv('DATASET_POSTGRES_PORT', '5432')}/"
                    f"{os.getenv('DATASET_POSTGRES_DB', 'heart_disease')}.heart_disease_processed"
                ),
                "test_size": 0.2,
                "split_random_state": 42,
                "stratify": True,
                "dataset_rows": len(df),
                "train_rows": len(X_train),
                "test_rows": len(X_test),
                "feature_count": X.shape[1],
                "dataset_sha256": dataset_sha256,
                "dataset_table": "heart_disease_processed",
                "selection_metric": "f1",
            })
            mlflow.set_tags({
                "model_family": "random_forest",
                "dataset_name": "UCI Heart Disease (id 45)",
                "dataset_fingerprint": dataset_sha256,
                "evaluation": "stratified_holdout",
            })
            mlflow.log_input(mlflow_dataset, context="training")
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            probabilities = model.predict_proba(X_test)[:, 1]
            metrics = {
                "accuracy": accuracy_score(y_test, pred),
                "f1": f1_score(y_test, pred, zero_division=0),
                "precision": precision_score(y_test, pred, zero_division=0),
                "recall": recall_score(y_test, pred, zero_division=0),
                "roc_auc": roc_auc_score(y_test, probabilities),
            }
            mlflow.log_metrics(metrics)
            matrix = confusion_matrix(y_test, pred)
            labels = sorted(y.unique())
            mlflow.log_dict(
                {"labels": [int(v) for v in labels], "matrix": matrix.tolist()},
                "evaluation/confusion_matrix.json",
            )
            fig, ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=matrix, display_labels=labels
            ).plot(ax=ax, cmap="Blues", colorbar=False)
            ax.set_title("Confusion matrix (test set)")
            fig.tight_layout()
            mlflow.log_figure(fig, "evaluation/confusion_matrix.png")
            plt.close(fig)
            mlflow.log_dict(
                classification_report(y_test, pred, output_dict=True, zero_division=0),
                "evaluation/classification_report.json",
            )
            model_info = mlflow.sklearn.log_model(
                model,
                name="model",
                input_example=X_train.head(5),
                serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            )
            runs.append({
                "run_id": run.info.run_id,
                "model_uri": model_info.model_uri,
                **params,
                **metrics,
            })

results = pd.DataFrame(runs).sort_values(
    ["f1", "roc_auc", "accuracy"], ascending=False
).reset_index(drop=True)
display(results)
assert len(results) == 20, f"Expected 20 runs, got {len(results)}"


/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference t

🏃 View run rf-trees-50-depth-none at: http://mlflow:5000/#/experiments/1/runs/e430c7c0075849b4a81687f91fb9e655
🧪 View experiment at: http://mlflow:5000/#/experiments/1


/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 04:21:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-trees-50-depth-5 at: http://mlflow:5000/#/experiments/1/runs/60d8fc248f6a479b84cbdad79c239849
🧪 View experiment at: http://mlflow:5000/#/experiments/1


/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 04:21:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 04:24:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-trees-250-depth-10 at: http://mlflow:5000/#/experiments/1/runs/2e66aad309a34140868da33de305764a
🧪 View experiment at: http://mlflow:5000/#/experiments/1


/python-env/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 04:24:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-trees-250-depth-15 at: http://mlflow:5000/#/experiments/1/runs/ae5a8dfbe17d41b48af046f4280c9e12
🧪 View experiment at: http://mlflow:5000/#/experiments/1


,run_id,model_uri,n_estimators,max_depth,min_samples_split,max_features,random_state,accuracy,f1,precision,recall,roc_auc
0,797eac92d27244be9b35df2ef0be52ff,models:/m-fd3a0613d26f4e1490b1b7cd644aaf9c,100,5.0,2,sqrt,42,0.916667,0.909091,0.925926,0.892857,0.958705
1,60d8fc248f6a479b84cbdad79c239849,models:/m-f0019092e9dc44b6b6114f54f9695083,50,5.0,2,sqrt,42,0.916667,0.909091,0.925926,0.892857,0.956473
2,e192aa06311b4071af9c7576cf0e8195,models:/m-7625c6f6e59146d18c20fbfd9cdf7ff1,150,5.0,2,sqrt,42,0.900000,0.892857,0.892857,0.892857,0.955357
3,e430c7c0075849b4a81687f91fb9e655,models:/m-fa380e5a3f44404088b78e988f80ec82,50,NaN,2,sqrt,42,0.883333,0.877193,0.862069,0.892857,0.949777
4,b2d3bae39d914d5683f3aa537e652730,models:/m-378544b3fd1f4ccfbff57891f142150d,50,15.0,2,sqrt,42,0.883333,0.877193,0.862069,0.892857,0.949777
5,5b3565942b4d46b4b3003d134241cc97,models:/m-62a08b44f9ac4909a4fa53d8228bc2e8,250,5.0,2,sqrt,42,0.883333,0.872727,0.888889,0.857143,0.954241
6,fd827bbf8fe645bb964ef2fd16b5ebe0,models:/m-a65d8137f7664765b45977512a3d50c0,200,5.0,2,sqrt,42,0.883333,0.872727,0.888889,0.857143,0.953125
7,adc2d2e08be0453498a8cba8144b9ad9,models:/m-70bafbc50c5c4fa98ca098fd2b24b3fd,50,10.0,2,sqrt,42,0.866667,0.862069,0.833333,0.892857,0.947545
8,df7f20d990d14e25816b9bc60c537ed4,models:/m-dff659de0cdf4a739b5f689ab3b126c2,200,NaN,2,sqrt,42,0.866667,0.857143,0.857143,0.857143,0.953125
9,2a0f8f208eae4ce1ae73d203d6773719,models:/m-353983f02fa44f609eb74684a6089279,200,15.0,2,sqrt,42,0.866667,0.857143,0.857143,0.857143,0.953125


## Register the best model

Selection uses F1; ROC AUC and accuracy break ties. The `champion` alias is updated to the model selected in this run. The `results` table keeps the ranking for review.

In [7]:
best = results.iloc[0]
registered_name = "HeartDiseaseClassifier"
client = MlflowClient(tracking_uri=TRACKING_URI, registry_uri=TRACKING_URI)
logged_model = client.get_logged_model(best.model_uri.removeprefix("models:/"))

if not client.search_registered_models(f"name='{registered_name}'"):
    client.create_registered_model(registered_name)

# Use the artifact location to avoid MLflow 3.5's failing models:/ lookup.
model_version = client.create_model_version(
    name=registered_name,
    source=logged_model.artifact_location,
    run_id=str(best.run_id),
    await_creation_for=30,
)
client.set_registered_model_alias(registered_name, "champion", str(model_version.version))

print("Best run:", best.run_id, "| F1:", best.f1)
print("Champion alias verified:",
      client.get_model_version_by_alias(registered_name, "champion").version)


2026/10/05 04:53:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 30 seconds for model version to finish creation. Model name: HeartDiseaseClassifier, version 2


Best run: 797eac92d27244be9b35df2ef0be52ff | F1: 0.9090909090909091
Champion alias verified: 2


The API loads `models:/HeartDiseaseClassifier@champion`. After the 20 runs finish, the final code cell registers the best model by F1 and sets the `champion` alias. Start predictions after that cell completes.

The training target is `0` for no heart disease and `1` for heart disease present. The API translates these classes into "No heart disease predicted" or "Heart disease predicted" and returns probabilities for both classes as percentages.